# Ensemble TWAS Weights via Stacked Regression

## Overview

`pecotmr` offers 10+ TWAS weight methods (SuSiE, LASSO, Elastic Net, mr.ash, BayesR, DPR, etc.), each suited to different genetic architectures. Picking one method is arbitrary; running all and testing each incurs a multiple-testing penalty. **Stacked regression** ([SR-TWAS; Dai et al.,2024](https://doi.org/10.1038/s41467-024-50983-w)) provides a principled alternative: learn one convex combination of methods per gene, producing a single weight vector for downstream TWAS testing with no method-selection bias.

This notebook:
1. Presents the mathematical framework and algorithm for ensemble weight learning
2. Benchmarks the approach via simulation using **genotypes** from LD sketch reference panels, so that realistic LD patterns are preserved
3. Produces diagnostic plots comparing CV $R^2$, MSE against ground truth, and MSE against observed $Y$

| Step | Description | Parallelism |
|------|-------------|-------------|
| `simulate_and_fit` | Load X from sketch LD, simulate Y, run methods + ensemble, save RDS | Per-region (`for_each`) |
| `analyze_results` | Load RDS files, compute metrics, generate diagnostic plots | Single task |

## Mathematical Framework

### Gene Expression Model

For a target gene $g$ with $p$ cis-SNPs, the expression level of individual $i$ is:

$$E_i = \mathbf{G}_i \mathbf{w} + \epsilon_i, \quad \epsilon_i \sim N(0, \sigma^2_\epsilon)$$

where $\mathbf{G}_i \in \mathbb{R}^{1 \times p}$ is the genotype vector and $\mathbf{w} \in \mathbb{R}^p$ is the true cis-eQTL effect-size vector.

### Base Models

We train $K$ prediction models $\{m_1, \ldots, m_K\}$, each producing a weight estimate $\hat{\mathbf{w}}_k \in \mathbb{R}^p$. Under $F$-fold cross-validation, the out-of-fold prediction for sample $i$ from method $k$ is:

$$\hat{E}_i^{(k)} = \mathbf{G}_i \hat{\mathbf{w}}_k^{(-f_i)}$$

where $f_i$ denotes the fold containing sample $i$ and $\hat{\mathbf{w}}_k^{(-f)}$ is trained on all samples except fold $f$. Stacking all $n$ out-of-fold predictions across $K$ methods yields the prediction matrix:

$$\mathbf{P} = \begin{pmatrix} \hat{E}_1^{(1)} & \cdots & \hat{E}_1^{(K)} \\ \vdots & \ddots & \vdots \\ \hat{E}_n^{(1)} & \cdots & \hat{E}_n^{(K)} \end{pmatrix} \in \mathbb{R}^{n \times K}$$

### Stacked Regression (Constrained QP)

We seek ensemble coefficients $\boldsymbol{\zeta} = (\zeta_1, \ldots, \zeta_K)^\top$ that minimize the prediction error of the convex combination:

$$\min_{\boldsymbol{\zeta}} \;\; \left\| \mathbf{E} - \sum_{k=1}^{K} \zeta_k \hat{\mathbf{E}}^{(k)} \right\|^2 = \left\| \mathbf{E} - \mathbf{P} \boldsymbol{\zeta} \right\|^2$$

$$\text{subject to} \quad \zeta_k \geq 0 \;\; \forall k, \qquad \sum_{k=1}^{K} \zeta_k = 1$$

This is a convex quadratic program solved via `quadprog::solve.QP` with:

$$\mathbf{D} = \mathbf{P}^\top \mathbf{P} + \lambda \mathbf{I}, \quad \mathbf{d} = \mathbf{P}^\top \mathbf{E}$$

where $\lambda = 10^{-8} \cdot \text{mean}(\text{diag}(\mathbf{P}^\top \mathbf{P}))$ provides ridge regularization for numerical stability.

### Ensemble Weight Vector

The final ensemble TWAS weight vector combines the full-data weight estimates:

$$\tilde{\mathbf{w}} = \sum_{k=1}^{K} \zeta_k^* \hat{\mathbf{w}}_k$$

### Performance Metrics

Two complementary evaluation criteria:

- **Against ground truth** (oracle): $\text{MSE}_{\text{truth}} = \frac{1}{n} \| \mathbf{G}\hat{\mathbf{w}} - \mathbf{G}\mathbf{w}_{\text{true}} \|^2$
- **Against observed $Y$** (practical): $R^2_Y = \text{cor}(Y,\; \mathbf{G}\hat{\mathbf{w}})^2$ and $\text{MSE}_Y = \frac{1}{n}\|Y - \mathbf{G}\hat{\mathbf{w}}\|^2$, where $Y = \mathbf{G}\mathbf{w}_{\text{true}} + \epsilon$

## Algorithm

The algorithm below shows the existing pecotmr cross-validation framework (Steps 1--5) and the **new ensemble learning extension** (Steps 6--9, in bold). Step numbers reference the mathematical formulation above.

```
Algorithm: TWAS Weight Estimation with Ensemble Learning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:  Genotype matrix G (n x p), expression vector E (n x 1),
        weight methods M = {m_1, ..., m_K}, number of CV folds F
Output: Per-method weights {w_k}, ensemble coefficients {zeta_k},
        combined weight vector w_tilde, CV performance metrics

── Existing Framework: twas_weights_cv() + twas_weights() ──────

1. Partition n samples into F folds: {S_1, ..., S_F}
2. For each fold f = 1, ..., F:
   For each method k = 1, ..., K:
     a. Train:   w_k^(-f)  <-  m_k(G[T_f,], E[T_f])
     b. Predict:  E_hat^(k)[S_f]  <-  G[S_f,] * w_k^(-f)
3. Assemble out-of-fold prediction matrix P  (n x K)
     P[i,k] = E_hat_i^(k) from the fold containing sample i
4. Compute per-method performance:
     R^2_k = cor(E, P[,k])^2    for k = 1, ..., K
5. Train final weights on full data:
     w_k  <-  m_k(G, E)    for all k

── NEW: Ensemble Learning via ensemble_weights() ──────────────

  6. Solve constrained QP for ensemble coefficients:
       D  <-  P^T P + lambda * I      (ridge regularization)
       d  <-  P^T E
       zeta*  <-  argmin  1/2 zeta^T D zeta - d^T zeta
                  s.t.  sum(zeta_k) = 1,  zeta_k >= 0  for all k
  7. Compute ensemble prediction:
       E_hat_ens  =  P zeta*
  8. Compute ensemble performance:
       R^2_ens  =  cor(E, E_hat_ens)^2
  9. Combine final weight vectors:
       w_tilde  =  sum_k  zeta_k * w_k

Return: {w_k}, {zeta_k}, w_tilde, {R^2_k}, R^2_ens
```

**Implementation**: Steps 1--5 are `pecotmr::twas_weights_cv()` and `pecotmr::twas_weights()`. Steps 6--9 are `ensemble_weights()` from `R/ensemble_weights.R`.

## Weight Methods

Two method sets are provided depending on computational budget:

### Expensive (default)

| # | Method | Type | pecotmr function | Key parameters |
|---|--------|------|------------------|----------------|
| 1 | SuSiE | Bayesian sparse (L0-like) | `susie_weights` | `refine=FALSE, init_L=5, max_L=10` |
| 2 | mr.ash | Bayesian adaptive shrinkage | `mrash_weights` | `init_prior_sd=TRUE` |
| 3 | BayesR | Bayesian mixture of normals | `bayes_r_weights` | default (`qgg`) |
| 4 | DPR | Dirichlet process regression | `dpr_weights` | default (`RcppDPR`) |
| 5 | LASSO | L1 penalized (glmnet) | `lasso_weights` | default |
| 6 | Elastic Net | L1+L2 penalized (glmnet) | `enet_weights` | default (`alpha=0.5`) |
| 7 | L0Learn | Best-subset approximation | `l0learn_weights` | default |

### Cheap (fast iteration)

| # | Method | Type | pecotmr function |
|---|--------|------|------------------|
| 1 | SuSiE | Bayesian sparse | `susie_weights` |
| 2 | mr.ash | Bayesian adaptive shrinkage | `mrash_weights` |
| 3 | LASSO | L1 penalized | `lasso_weights` |
| 4 | Elastic Net | L1+L2 penalized | `enet_weights` |

## Simulation Design

### Genotypes

Each replicate uses **genotypes** loaded from LD sketch reference panels via `pecotmr::load_LD_matrix(meta, region, return_genotype = TRUE)`. Preserving LD patterns is critical for faithfully benchmarking TWAS weight methods — no synthetic genotype simulation can reproduce the complex correlation structure of actual genomic data.

The pipeline requires two input files:

1. **`ld_meta_file`** — Sketch LD metadata TSV pointing to plink2 files (one row per chromosome, `start=0, end=0` sentinel). Generated by the `rss_ld_sketch` pipeline. Example:
   ```
   #chrom  start  end  path
   22      0      0    ADSP.R5.EUR.chr22
   ```
   The `path` column is the plink2 file prefix (resolved relative to the TSV directory). The corresponding `.pgen`, `.pvar`, `.psam`, `.afreq` files must exist.

2. **`ld_block_file`** — BED or TSV file defining LD block boundaries (one row per block). Each block becomes a replicate region. Example (BED format, 0-based half-open):
   ```
   chr22  16051249  17614263
   chr22  17614263  18834263
   ...
   ```
   Or TSV with `#chrom  start  end` header. These define the genomic windows passed to `load_LD_matrix()` to extract variants per region.

### Phenotypes

- **Sparse**: `n_sparse` variants with large effects
- **Oligogenic**: `n_oligogenic` variants with moderate effects
- **Infinitesimal**: `n_inf` variants with small effects
- Total heritability: $h^2_g$ (default 0.2)

### Replicates

Each genomic region in the `ld_block_file` defines one replicate. The `n_seeds` parameter controls the number of independent phenotype realizations per region (different random seeds produce different causal architectures on the same genotype matrix).

### Data Saving

Set `save_data = TRUE` to save the raw genotype matrix, simulated phenotype, and ground-truth causal information in a separate `data.rds` file per replicate. This is useful for debugging but produces large files (one per region x seed). Disabled by default.

### Evaluation Metrics

Two complementary metrics per replicate:

1. **Against ground truth** — MSE of predicted genetic component vs. true genetic component: $\frac{1}{n}\|\mathbf{G}\hat{\mathbf{w}} - \mathbf{G}\mathbf{w}_{\text{true}}\|^2$. This measures how well the method recovers the true signal.

2. **Against observed $Y$** — $R^2$ and MSE of predictions vs. observed phenotype $Y = \mathbf{G}\mathbf{w}_{\text{true}} + \epsilon$. This is the practical metric: how well would the weights predict expression in a held-out sample?

## Usage

### Quick test (cheap methods, few regions)

In [ ]:
sos run pipeline/ensemble_twas_weights.ipynb simulate_and_fit \
    --ld-meta-file data/ld_meta_file.tsv \
    --ld-block-file /restricted/projectnb/casa/oaolayin/ROSMAP_NIA_geno/EUR_LD_blocks.bed \
    --chrom 21 \
    --n-seeds 1 \
    --method-set cheap \
    -J 20

### Full benchmark (expensive methods, 10 seeds per region)

In [ ]:
sos run pipeline/ensemble_twas_weights.ipynb simulate_and_fit \
    --ld-meta-file /path/to/sketch_ld_meta.tsv \
    --ld-block-file /path/to/EUR_LD_blocks.bed \
    --chrom 22 \
    --n-seeds 10 \
    --method-set expensive \
    -J 40

### With data saving (for debugging)

In [ ]:
sos run pipeline/ensemble_twas_weights.ipynb simulate_and_fit \
    --ld-meta-file /path/to/sketch_ld_meta.tsv \
    --ld-block-file /path/to/EUR_LD_blocks.bed \
    --chrom 22 \
    --n-seeds 1 \
    --save-data TRUE \
    -J 20


### Analysis and plots

In [ ]:
sos run pipeline/ensemble_twas_weights.ipynb analyze_results \
    --cwd /restricted/projectnb/xqtl/jaempawi/xqtl_protocol/toy_example/output/ensemble_twas \
    --ld-meta-file /restricted/projectnb/xqtl/jaempawi/xqtl_protocol/toy_example/output/rss_ld_sketch/sketch_ld_meta.tsv \
    --ld-block-file /restricted/projectnb/casa/oaolayin/ROSMAP_NIA_geno/EUR_LD_blocks.bed

# Pipeline Implementation

In [ ]:
[global]
# Output directory
parameter: cwd = path("output/ensemble_twas")

# Path to ensemble_weights.R source file
parameter: ensemble_weights_source = path("R/ensemble_weights.R")

# ── Genotype source ──────────────────────────────────────────
# Sketch LD metadata TSV (plink2 format: chrom, start=0, end=0, path=prefix)
# Generated by the rss_ld_sketch pipeline. Used by load_LD_matrix() to load genotypes.
parameter: ld_meta_file = path("")
# LD block file defining replicate regions (BED or TSV with chrom/start/end)
# Each block becomes one replicate. Can be EUR_LD_blocks.bed or similar.
parameter: ld_block_file = path("")
# Chromosome filter (0 = all chromosomes)
parameter: chrom = 0
# Max variants to subsample per region (0 = no limit)
parameter: max_variants = 0

# ── Method selection ─────────────────────────────────────────
# "expensive": SuSiE, mr.ash, BayesR, DPR, LASSO, Elastic Net, L0Learn
# "cheap":     SuSiE, mr.ash, LASSO, Elastic Net
parameter: method_set = "expensive"

# ── Simulation parameters ────────────────────────────────────
parameter: h2g = 0.2
parameter: n_sparse = 3
parameter: n_oligogenic = 5
parameter: n_inf = 15
parameter: cv_folds = 5
parameter: max_cv_variants = 500

# ── Replicate control ────────────────────────────────────────
parameter: n_seeds = 1
parameter: seed_base = 42

# ── Data saving (for debugging) ──────────────────────────────
# When TRUE, saves raw genotype matrix X, phenotype y, and sim object
# as a separate data.rds per replicate. Produces large files.
parameter: save_data = False

# ── Cluster resources ────────────────────────────────────────
parameter: job_size = 1
parameter: walltime = "4:00:00"
parameter: mem = "16G"
parameter: numThreads = 4

import os
cwd = path(f'{cwd:a}')

def _read_ld_blocks(block_file, chrom_filter):
    """Parse LD block file (BED or TSV with chrom/start/end columns).
    Returns list of region dicts with id, chrom, start, end, region string.
    Supports BED (0-based half-open) and TSV with #chrom header.
    """
    regions = []
    with open(block_file) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith('#chrom') or line.startswith('chrom'):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            c = parts[0]
            if not c.startswith('chr'):
                c = f'chr{c}'
            try:
                cnum = int(c.replace('chr', ''))
            except ValueError:
                continue
            if not (1 <= cnum <= 22):
                continue
            if chrom_filter != 0 and cnum != chrom_filter:
                continue
            start, end = int(parts[1]), int(parts[2])
            regions.append({
                'id': f'{c}_{start}_{end}',
                'chrom': c,
                'start': start,
                'end': end,
                'region': f'{c}:{start}-{end}'
            })
    return regions

# Validate inputs
if not str(ld_meta_file) or not os.path.isfile(str(ld_meta_file)):
    raise ValueError("ld_meta_file is required — must point to a sketch LD metadata TSV (plink2 format)")
if not str(ld_block_file) or not os.path.isfile(str(ld_block_file)):
    raise ValueError("ld_block_file is required — must point to an LD block BED/TSV defining replicate regions")

# Build replicate list: regions x seeds
_base_regions = _read_ld_blocks(str(ld_block_file), chrom)
if not _base_regions:
    raise ValueError(f"No regions found in {ld_block_file} for chrom={chrom}")

replicates = []
for seed_idx in range(n_seeds):
    for i, r in enumerate(_base_regions):
        rep = dict(r)
        rep['seed'] = seed_base + seed_idx * len(_base_regions) + i
        rep['rep_id'] = f'{r["id"]}_seed{seed_idx}'
        replicates.append(rep)

print(f"  {len(_base_regions)} regions x {n_seeds} seeds = {len(replicates)} replicates")

### `simulate_and_fit`

For each replicate (region x seed):
1. Load genotype matrix $\mathbf{G}$ from LD sketch via `pecotmr::load_LD_matrix()` using the sketch LD metadata and the region defined by the LD block file
2. Simulate expression $E$ via `simxQTL::generate_cis_qtl_data()` with mixed architecture on top of $\mathbf{G}$
3. Run `pecotmr::twas_weights_pipeline()` with selected methods and 5-fold CV
4. Call `ensemble_weights()` to learn $\boldsymbol{\zeta}$ and combine weights
5. Compute metrics against ground truth and against observed $Y$
6. Save results as RDS; optionally save raw data (X, y, sim) as separate `data.rds` when `--save-data TRUE`

In [ ]:
[simulate_and_fit]
input: for_each = "replicates"
output_files = [f'{cwd}/{_replicates["rep_id"]}/results.rds']
if save_data:
    output_files.append(f'{cwd}/{_replicates["rep_id"]}/data.rds')
output: output_files
# Pre-compute data path in Python so the R script can reference it unconditionally.
# (Using ${_output[1]} directly fails with IndexError when save_data=False
#  because SoS's ${} substitution is Python-eager, regardless of R control flow.)
_data_path = f'{cwd}/{_replicates["rep_id"]}/data.rds' if save_data else ''
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads
R: expand = "${ }", stderr = f'{_output[0]:n}.stderr', stdout = f'{_output[0]:n}.stdout'

    library(pecotmr)
    library(simxQTL)

    # Source the ensemble_weights function
    source("${ensemble_weights_source}")

    set.seed(${_replicates['seed']})
    cat("Replicate: ${_replicates['rep_id']}\n")
    cat("Seed: ${_replicates['seed']}\n")
    cat("Region: ${_replicates['region']}\n")

    # ── 1. Load genotype from LD sketch ─────────────────────
    # ld_meta_file points to sketch LD plink2 files (whole-chromosome)
    # region from ld_block_file selects a specific LD block window
    ld_data <- load_LD_matrix(
        "${ld_meta_file}",
        region = "${_replicates['region']}",
        return_genotype = TRUE
    )
    X <- ld_data$LD_matrix  # genotype matrix (B x p) from sketch
    X <- scale(X)
    X[is.na(X)] <- 0  # handle monomorphic variants after scaling
    n <- nrow(X); p <- ncol(X)
    cat(sprintf("Genotype matrix: %d samples x %d variants\n", n, p))

    # Optionally subsample variants
    max_var <- ${max_variants}
    if (max_var > 0 && p > max_var) {
        keep_idx <- sort(sample(p, max_var))
        X <- X[, keep_idx, drop = FALSE]
        p <- ncol(X)
        cat(sprintf("Subsampled to %d variants\n", p))
    }

    # ── 2. Simulate phenotype on genotypes ───────────────────
    sim <- generate_cis_qtl_data(X, h2g = ${h2g},
                                  n_sparse = ${n_sparse},
                                  n_oligogenic = ${n_oligogenic},
                                  n_inf = ${n_inf})
    y <- sim$y  # observed Y = G*w_true + epsilon
    cat(sprintf("Realized h2g: %.4f\n", sim$h2g))

    # ── 2b. Optionally save raw data for debugging ───────────────
    data_path <- "${_data_path}"
    if (nzchar(data_path)) {
        data_out <- list(
            X = X,
            y = y,
            sim = sim,
            region = "${_replicates['region']}",
            rep_id = "${_replicates['rep_id']}",
            seed = ${_replicates['seed']},
            variant_info = ld_data$ref_panel
        )
        dir.create(dirname(data_path), recursive = TRUE, showWarnings = FALSE)
        saveRDS(data_out, data_path)
        cat("Saved raw data:", data_path, "\n")
        rm(data_out)  # free memory
    }

    # ── 3. Define weight methods ─────────────────────────────────
    method_set <- "${method_set}"
    if (method_set == "expensive") {
        weight_methods <- list(
            susie_weights   = list(refine = FALSE, init_L = 5, max_L = 10),
            mrash_weights   = list(init_prior_sd = TRUE, max.iter = 100),
            bayes_r_weights = list(),
            dpr_weights     = list(),
            lasso_weights   = list(),
            enet_weights    = list(),
            l0learn_weights = list()
        )
    } else {
        weight_methods <- list(
            susie_weights = list(refine = FALSE, init_L = 5, max_L = 10),
            mrash_weights = list(init_prior_sd = TRUE, max.iter = 100),
            lasso_weights = list(),
            enet_weights  = list()
        )
    }

    # ── 4. Run TWAS weights pipeline (CV + full-data weights) ────
    # Note: twas_weights_pipeline() in pecotmr >=0.5 does NOT accept num_threads.
    cat("Running twas_weights_pipeline with", length(weight_methods), "methods...\n")
    res <- tryCatch(
        twas_weights_pipeline(
            X, y,
            cv_folds = ${cv_folds},
            weight_methods = weight_methods,
            max_cv_variants = ${max_cv_variants}
        ),
        error = function(e) {
            cat("Pipeline error:", conditionMessage(e), "\n")
            NULL
        }
    )

    if (is.null(res)) {
        cat("Pipeline failed — exiting non-zero so SoS flags this replicate\n")
        saveRDS(list(results = NULL, ensemble = NULL,
                     truth = list(rep_id = "${_replicates['rep_id']}",
                                  seed = ${_replicates['seed']},
                                  error = TRUE)),
                "${_output[0]}")
        quit(save = "no", status = 1)
    }

    # ── 5. Ensemble learning (Steps 6-9 in Algorithm) ────────────
    cat("Computing ensemble weights...\n")
    ens <- tryCatch(
        ensemble_weights(
            cv_results = res$twas_cv_result,
            Y = y,
            twas_weight_list = res$twas_weights
        ),
        error = function(e) {
            cat("Ensemble error:", conditionMessage(e), "\n")
            NULL
        }
    )

    # ── 6. Compute metrics ───────────────────────────────────────
    # simxQTL returns sim$beta as a length-p vector with zeros at non-causal
    # positions (causal positions indicated by sim$sparse_indices /
    # sim$oligogenic_indices / sim$infinitesimal_indices for forensics).
    # Use sim$beta directly.
    if (!is.null(sim$beta) && length(sim$beta) == p) {
        true_beta <- as.numeric(sim$beta)
    } else {
        warning("sim$beta missing or wrong length (", length(sim$beta),
                " vs p=", p, "); defaulting to zero vector")
        true_beta <- rep(0, p)
    }
    true_genetic <- as.vector(X %*% true_beta)

    # Helper: compute metrics for a weight vector
    compute_metrics <- function(w_hat) {
        if (is.matrix(w_hat)) w_hat <- w_hat[, 1]
        pred <- as.vector(X %*% w_hat)
        # Against ground truth
        mse_truth <- mean((pred - true_genetic)^2)
        # Against observed Y
        mse_y <- mean((pred - y)^2)
        rsq_y <- if (sd(pred) > 0) cor(pred, y)^2 else 0
        c(mse_truth = mse_truth, mse_y = mse_y, rsq_y = rsq_y)
    }

    # Per-method metrics
    method_metrics <- sapply(names(res$twas_weights), function(wname) {
        compute_metrics(res$twas_weights[[wname]])
    })
    colnames(method_metrics) <- gsub("_weights$", "", colnames(method_metrics))

    # Ensemble metrics
    ensemble_metrics <- c(mse_truth = NA, mse_y = NA, rsq_y = NA)
    if (!is.null(ens) && !is.null(ens$ensemble_twas_weights)) {
        ensemble_metrics <- compute_metrics(ens$ensemble_twas_weights)
    }

    # ── 7. Save complete results ─────────────────────────────────
    truth <- list(
        causal_sparse = sim$sparse_indices,
        causal_oligo = sim$oligogenic_indices,
        causal_inf = sim$infinitesimal_indices,
        true_beta = true_beta,
        true_y = y,
        h2g = sim$h2g,
        rep_id = "${_replicates['rep_id']}",
        region = "${_replicates['region']}",
        seed = ${_replicates['seed']},
        n = n, p = p,
        method_metrics = method_metrics,
        ensemble_metrics = ensemble_metrics
    )

    out <- list(results = res, ensemble = ens, truth = truth)
    dir.create(dirname("${_output[0]}"), recursive = TRUE, showWarnings = FALSE)
    saveRDS(out, "${_output[0]}")
    cat("Saved:", "${_output[0]}", "\n")
    cat("Ensemble CV R^2:", if (!is.null(ens)) round(ens$ensemble_performance["rsq"], 4) else NA, "\n")
    cat("Ensemble MSE (truth):", round(ensemble_metrics["mse_truth"], 6), "\n")
    cat("Ensemble MSE (obs Y):", round(ensemble_metrics["mse_y"], 6), "\n")

### `analyze_results`

Load all per-replicate RDS files and produce:
1. **CV $R^2$ boxplot** — all methods + ensemble, ordered by median
2. **MSE against ground truth boxplot** — $\|\mathbf{G}\hat{\mathbf{w}} - \mathbf{G}\mathbf{w}_{\text{true}}\|^2 / n$
3. **MSE against observed $Y$ boxplot** — $\|Y - \mathbf{G}\hat{\mathbf{w}}\|^2 / n$
4. **$R^2$ against observed $Y$ boxplot** — $\text{cor}(Y, \mathbf{G}\hat{\mathbf{w}})^2$
5. **Coefficient heatmap** — ensemble $\zeta_k$ across replicates
6. **Scatter plot** — ensemble $R^2$ vs. best single-method $R^2$
7. **Summary CSV** — one row per replicate with all metrics

In [ ]:
[analyze_results]
parameter: results_dir = path(f'{cwd}')
output: f'{results_dir}/diagnostics.pdf', f'{results_dir}/summary_table.csv'
task: trunk_workers = 1, walltime = "1:00:00", mem = "8G", cores = 1
R: expand = "${ }", stderr = f'{results_dir}/analyze.stderr', stdout = f'{results_dir}/analyze.stdout'

    library(ggplot2)
    library(tidyr)
    library(dplyr)

    results_dir <- "${results_dir}"

    # ── 1. Load all RDS files ────────────────────────────────────
    rds_files <- list.files(results_dir, pattern = "results\\.rds$",
                            recursive = TRUE, full.names = TRUE)
    cat(sprintf("Found %d RDS files\n", length(rds_files)))

    all_data <- lapply(rds_files, function(f) {
        tryCatch(readRDS(f), error = function(e) NULL)
    })
    all_data <- all_data[!sapply(all_data, is.null)]
    all_data <- all_data[!sapply(all_data, function(x) isTRUE(x$truth$error))]
    cat(sprintf("%d successful replicates\n", length(all_data)))

    if (length(all_data) == 0) stop("No valid results found")

    # ── 2. Extract metrics ───────────────────────────────────────
    metrics_list <- lapply(seq_along(all_data), function(i) {
        x <- all_data[[i]]
        rep_id <- x$truth$rep_id
        seed <- x$truth$seed
        region <- x$truth$region

        # CV R^2 from ensemble output
        cv_rsq <- if (!is.null(x$ensemble)) x$ensemble$method_performance else NULL
        ens_cv_rsq <- if (!is.null(x$ensemble)) x$ensemble$ensemble_performance["rsq"] else NA
        coefs <- if (!is.null(x$ensemble)) x$ensemble$method_coef else NULL

        # Method metrics matrix (3 rows x K methods): mse_truth, mse_y, rsq_y
        mm <- x$truth$method_metrics
        em <- x$truth$ensemble_metrics

        list(
            rep_id = rep_id, seed = seed, region = region,
            cv_rsq = cv_rsq, ens_cv_rsq = ens_cv_rsq,
            method_coef = coefs,
            method_metrics = mm, ensemble_metrics = em
        )
    })

    # ── 3. Build tidy data frames ────────────────────────────────

    # Helper to build long-format metric df
    build_metric_df <- function(metric_name) {
        do.call(rbind, lapply(metrics_list, function(m) {
            if (is.null(m$method_metrics)) return(NULL)
            methods <- colnames(m$method_metrics)
            vals <- m$method_metrics[metric_name, ]
            method_df <- data.frame(
                rep_id = m$rep_id, method = methods,
                value = as.numeric(vals), stringsAsFactors = FALSE)
            ens_df <- data.frame(
                rep_id = m$rep_id, method = "ensemble",
                value = as.numeric(m$ensemble_metrics[metric_name]),
                stringsAsFactors = FALSE)
            rbind(method_df, ens_df)
        }))
    }

    # CV R^2 (from ensemble output, not method_metrics)
    cv_rsq_df <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$cv_rsq)) return(NULL)
        method_df <- data.frame(
            rep_id = m$rep_id, method = names(m$cv_rsq),
            value = as.numeric(m$cv_rsq), stringsAsFactors = FALSE)
        ens_df <- data.frame(
            rep_id = m$rep_id, method = "ensemble",
            value = as.numeric(m$ens_cv_rsq), stringsAsFactors = FALSE)
        rbind(method_df, ens_df)
    }))

    mse_truth_df <- build_metric_df("mse_truth")
    mse_y_df     <- build_metric_df("mse_y")
    rsq_y_df     <- build_metric_df("rsq_y")

    # Coefficient matrix for heatmap
    coef_mat <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$method_coef)) return(NULL)
        m$method_coef
    }))
    if (!is.null(coef_mat)) {
        rownames(coef_mat) <- sapply(metrics_list[!sapply(metrics_list,
            function(m) is.null(m$method_coef))], function(m) m$rep_id)
    }

    # Scatter data: ensemble vs best single method (using rsq_y)
    scatter_df <- do.call(rbind, lapply(metrics_list, function(m) {
        if (is.null(m$method_metrics)) return(NULL)
        rsq_vals <- m$method_metrics["rsq_y", ]
        data.frame(
            rep_id = m$rep_id,
            best_single_rsq = max(rsq_vals, na.rm = TRUE),
            best_method = names(which.max(rsq_vals)),
            ensemble_rsq = as.numeric(m$ensemble_metrics["rsq_y"]),
            stringsAsFactors = FALSE
        )
    }))

    # ── 4. Diagnostic plots ──────────────────────────────────────

    # Boxplot helper
    make_boxplot <- function(df, y_label, title, descending = FALSE) {
        if (is.null(df) || nrow(df) == 0) return(NULL)
        ord <- df %>% group_by(method) %>%
            summarise(med = median(value, na.rm = TRUE))
        if (descending) ord <- ord %>% arrange(desc(med)) else ord <- ord %>% arrange(med)
        df$method <- factor(df$method, levels = ord$method)
        ggplot(df, aes(x = method, y = value,
                       fill = ifelse(method == "ensemble", "Ensemble", "Single"))) +
            geom_boxplot(outlier.size = 0.8) +
            scale_fill_manual(values = c("Ensemble" = "#E41A1C", "Single" = "#377EB8"), name = "") +
            labs(title = title, x = "Method", y = y_label) +
            theme_bw(base_size = 12) +
            theme(axis.text.x = element_text(angle = 45, hjust = 1))
    }

    pdf("${_output[0]}", width = 12, height = 8)

    # Plot 1: CV R^2
    p <- make_boxplot(cv_rsq_df, expression(CV~R^2),
                      "Cross-Validation R-squared by Method")
    if (!is.null(p)) print(p)

    # Plot 2: MSE against ground truth
    p <- make_boxplot(mse_truth_df, "MSE (vs truth)",
                      "MSE Against Ground Truth (True Genetic Component)",
                      descending = TRUE)
    if (!is.null(p)) print(p)

    # Plot 3: MSE against observed Y
    p <- make_boxplot(mse_y_df, "MSE (vs observed Y)",
                      "MSE Against Observed Phenotype Y",
                      descending = TRUE)
    if (!is.null(p)) print(p)

    # Plot 4: R^2 against observed Y
    p <- make_boxplot(rsq_y_df, expression(R^2~(vs~observed~Y)),
                      "R-squared Against Observed Phenotype Y")
    if (!is.null(p)) print(p)

    # Plot 5: Coefficient heatmap
    if (!is.null(coef_mat) && nrow(coef_mat) > 1) {
        coef_long <- as.data.frame(coef_mat) %>%
            mutate(replicate = rownames(coef_mat)) %>%
            pivot_longer(-replicate, names_to = "method", values_to = "zeta")
        p5 <- ggplot(coef_long, aes(x = method, y = replicate, fill = zeta)) +
            geom_tile() +
            scale_fill_gradient(low = "white", high = "#E41A1C",
                                name = expression(zeta[k]), limits = c(0, 1)) +
            labs(title = "Ensemble Coefficients Across Replicates",
                 x = "Method", y = "Replicate") +
            theme_bw(base_size = 10) +
            theme(axis.text.x = element_text(angle = 45, hjust = 1),
                  axis.text.y = element_text(size = 6))
        print(p5)
    }

    # Plot 6: Ensemble vs best single-method scatter
    if (!is.null(scatter_df) && nrow(scatter_df) > 0) {
        p6 <- ggplot(scatter_df, aes(x = best_single_rsq, y = ensemble_rsq)) +
            geom_point(alpha = 0.6, size = 2, color = "#377EB8") +
            geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "grey40") +
            labs(title = "Ensemble vs Best Single Method (R-squared vs Observed Y)",
                 x = expression(Best~single~method~R^2),
                 y = expression(Ensemble~R^2)) +
            theme_bw(base_size = 12) +
            coord_equal()
        print(p6)
    }

    dev.off()
    cat("Saved plots:", "${_output[0]}", "\n")

    # ── 5. Summary table ─────────────────────────────────────────
    summary_rows <- lapply(metrics_list, function(m) {
        row <- data.frame(
            rep_id = m$rep_id,
            region = m$region,
            seed = m$seed,
            ensemble_cv_rsq = as.numeric(m$ens_cv_rsq),
            ensemble_mse_truth = as.numeric(m$ensemble_metrics["mse_truth"]),
            ensemble_mse_y = as.numeric(m$ensemble_metrics["mse_y"]),
            ensemble_rsq_y = as.numeric(m$ensemble_metrics["rsq_y"]),
            stringsAsFactors = FALSE
        )
        # Per-method metrics
        if (!is.null(m$method_metrics)) {
            for (mn in colnames(m$method_metrics)) {
                row[[paste0(mn, "_mse_truth")]] <- m$method_metrics["mse_truth", mn]
                row[[paste0(mn, "_mse_y")]] <- m$method_metrics["mse_y", mn]
                row[[paste0(mn, "_rsq_y")]] <- m$method_metrics["rsq_y", mn]
            }
        }
        if (!is.null(m$cv_rsq)) {
            for (mn in names(m$cv_rsq)) {
                row[[paste0(mn, "_cv_rsq")]] <- m$cv_rsq[[mn]]
            }
        }
        if (!is.null(m$method_coef)) {
            for (mn in names(m$method_coef)) {
                row[[paste0("zeta_", mn)]] <- m$method_coef[[mn]]
            }
        }
        row
    })
    summary_df <- bind_rows(summary_rows)

    # Best single method and improvement
    method_rsq_cols <- grep("_rsq_y$", names(summary_df), value = TRUE)
    method_rsq_cols <- setdiff(method_rsq_cols, "ensemble_rsq_y")
    if (length(method_rsq_cols) > 0) {
        summary_df$best_method_rsq_y <- apply(summary_df[method_rsq_cols], 1, max, na.rm = TRUE)
        summary_df$best_method <- apply(summary_df[method_rsq_cols], 1, function(r) {
            gsub("_rsq_y$", "", names(which.max(r)))
        })
        summary_df$rsq_y_improvement <- summary_df$ensemble_rsq_y - summary_df$best_method_rsq_y
    }

    write.csv(summary_df, "${_output[1]}", row.names = FALSE)
    cat("Saved summary:", "${_output[1]}", "\n")

    # Print summary statistics
    cat("\n=== Summary ===", "\n")
    cat(sprintf("Replicates: %d\n", nrow(summary_df)))
    cat(sprintf("Mean ensemble CV R^2:       %.4f\n", mean(summary_df$ensemble_cv_rsq, na.rm = TRUE)))
    cat(sprintf("Mean ensemble MSE (truth):  %.6f\n", mean(summary_df$ensemble_mse_truth, na.rm = TRUE)))
    cat(sprintf("Mean ensemble MSE (obs Y):  %.6f\n", mean(summary_df$ensemble_mse_y, na.rm = TRUE)))
    cat(sprintf("Mean ensemble R^2 (obs Y):  %.4f\n", mean(summary_df$ensemble_rsq_y, na.rm = TRUE)))
    if ("rsq_y_improvement" %in% names(summary_df)) {
        cat(sprintf("Ensemble >= best in %.1f%% of replicates (R^2 vs Y)\n",
            100 * mean(summary_df$rsq_y_improvement >= 0, na.rm = TRUE)))
    }

## Expected Results

With default simulation parameters ($h^2_g = 0.2$, mixed architecture) on genotype regions:

- **CV $R^2$**: Range depends on region size ($p$) and sample size ($n$). Ensemble should match or slightly exceed the best single method.
- **MSE against truth vs. MSE against $Y$**: MSE against truth isolates weight estimation quality (unaffected by noise). MSE against $Y$ includes irreducible noise $\sigma^2_\epsilon$, so it is always larger.
- **Coefficient patterns**: Sparse architectures favor SuSiE/LASSO; polygenic architectures favor BayesR/DPR; the ensemble adapts by assigning larger $\zeta_k$ to the method best suited to each region's architecture.
- **Scatter plot**: Most points should lie on or above the diagonal, indicating ensemble matches or improves on the best single method.